In [4]:
import sys
from pathlib import Path

ROOT = Path("/home/aj/redesigned-octo-couscous") 
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import dct
import torch
from transformers import GPT2Tokenizer
from tensor_bilinear_attn_model import TensorBlockSpan, TensorMiddleSpan, load_tensor_gpt, mixed_difference

device = torch.device("cuda")

model, config, metadata = load_tensor_gpt("Elriggs/gpt2-bilinear-sqrd-attn-18l-9h-1152embd", device)
model.requires_grad_(False)
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
import random
import csv

def read_texts(train_count, heldout_count, transfer_count):
    with (ROOT / "harmful_behaviors.csv").open() as handle:
        behavior = list(dict.fromkeys(row["target"].strip() for row in csv.DictReader(handle)))
    with (ROOT / "harmful_strings.csv").open() as handle:
        transfer = list(dict.fromkeys(row["target"].strip() for row in csv.DictReader(handle)))
    random.Random(1729).shuffle(behavior)
    random.Random(2718).shuffle(transfer)
    return (behavior[:train_count], behavior[train_count:train_count + heldout_count],
            transfer[:transfer_count])
def collect_mlp_inputs(model, tokenizer, texts, layer, sequence_length, batch_size):
    inputs = []
    for start in range(0, len(texts), batch_size):
        encoded = tokenizer(texts[start:start + batch_size], return_tensors="pt", truncation=True,
                            padding="max_length", max_length=sequence_length).input_ids.to(next(model.parameters()).device)
        with torch.no_grad():
            inputs.append(model.trace(encoded)["mlp_inputs"][layer].float())
    return torch.cat(inputs)
def collect_embedding_inputs(model, tokenizer, texts, sequence_length, batch_size):
    inputs = []
    for start in range(0, len(texts), batch_size):
        encoded = tokenizer(texts[start:start + batch_size], return_tensors="pt", truncation=True,
                            padding="max_length", max_length=sequence_length).input_ids.to(next(model.parameters()).device)
        with torch.no_grad():
            inputs.append(model.embedding_residual(encoded).float())
    return torch.cat(inputs)
def collect_middle_inputs(model, tokenizer, texts, source_layer, sequence_length):
    values, initial_values, first_values = [], [], []
    for text in texts:
        encoded = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length",
                            max_length=sequence_length).input_ids.to(next(model.parameters()).device)
        with torch.no_grad():
            state = model.state_before_block(encoded, source_layer)
        values.append(state["values"].float())
        initial_values.append(state["initial_values"].float())
        first_values.append(state["first_values"].float())
    return torch.cat(values), torch.cat(initial_values), torch.cat(first_values)